In [1]:
%pip install pandas numpy duckdb pyarrow scikit-learn xgboost matplotlib seaborn ipykernel


[notice] A new release of pip is available: 24.2 -> 26.2.2
[notice] To update, run: pip3.12 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import duckdb
from pathlib import Path

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
import sys

print("Python version:", sys.version)
print("Python executable:", sys.executable)
print("Pandas version:", pd.__version__)
print("DuckDB version:", duckdb.__version__)

Python version: 3.12.5 (v3.12.5:ff3bc82f7c9, Aug  7 2024, 05:32:06) [Clang 13.0.0 (clang-1300.0.29.30)]
Python executable: /usr/local/bin/python3.12
Pandas version: 3.0.6
DuckDB version: 2.0.0


In [4]:
# Step 4 - Locate and inspect the wildfire dataset

# Find the dataset
candidates = [
    Path("data/processed/master_training_dataset.parquet"),
    Path("../data/processed/master_training_dataset.parquet"),
    Path("data/master_training_dataset.parquet"),
    Path("../data/master_training_dataset.parquet")
]

data_path = next(
    (p.resolve() for p in candidates if p.is_file()),
    None
)

if data_path is None:
    raise FileNotFoundError("Dataset not found. Check the file location.")

print("Dataset found:", data_path)
print("File size (GB):", round(data_path.stat().st_size / 1e9, 2))

# Connect to DuckDB
con = duckdb.connect()

# Inspect dataset columns and data types
schema = con.execute(
    "DESCRIBE SELECT * FROM read_parquet(?)",
    [str(data_path)]
).df()

display(schema)

Dataset found: /Users/amsal/Documents/GitHub/wildfire-prediction-ca/data/processed/master_training_dataset.parquet
File size (GB): 1.13


,column_name,column_type,null,key,default,extra
0,cell_id,INTEGER,YES,None,None,None
1,valid_time,TIMESTAMP_NS,YES,None,None,None
2,number,FLOAT,YES,None,None,None
3,t2m,FLOAT,YES,None,None,None
4,rh2m,FLOAT,YES,None,None,None
5,wind_speed_10m,FLOAT,YES,None,None,None
6,tp,FLOAT,YES,None,None,None
7,ndvi,FLOAT,YES,None,None,None
8,evi,FLOAT,YES,None,None,None
9,longitude,FLOAT,YES,None,None,None


In [5]:
# Step 17 - Validate corrected master dataset

validation = con.execute("""
    WITH daily_counts AS (
        SELECT
            cell_id,
            valid_time,
            COUNT(*) AS n
        FROM read_parquet(?)
        GROUP BY cell_id, valid_time
    )
    SELECT
        COUNT(*) AS unique_cell_days,
        SUM(n) AS total_rows,
        SUM(n - 1) AS duplicate_rows,
        MAX(n) AS max_repeats
    FROM daily_counts
""", [str(data_path)]).df()

display(validation)

,unique_cell_days,total_rows,duplicate_rows,max_repeats
0,83428616,83428616.0,0.0,1


In [6]:
# Step 18 - Validate weather coverage and wildfire labels

coverage_check = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT cell_id) AS total_cells,
        MIN(valid_time) AS earliest_date,
        MAX(valid_time) AS latest_date,

        COUNT(t2m) AS temperature_available,
        COUNT(rh2m) AS humidity_available,
        COUNT(wind_speed_10m) AS wind_available,
        COUNT(tp) AS precipitation_available,

        COUNT(*) FILTER (
            WHERE fire_occurred = 1
        ) AS fire_rows,

        COUNT(*) FILTER (
            WHERE fire_occurred = 0
        ) AS no_fire_rows,

        COUNT(*) FILTER (
            WHERE fire_occurred IS NULL
               OR fire_occurred NOT IN (0, 1)
        ) AS invalid_labels

    FROM read_parquet(?)
    WHERE valid_time >= '2024-01-01'
      AND valid_time < '2024-02-01'
""", [str(data_path)]).df()

display(coverage_check)

total = coverage_check["total_rows"].iloc[0]

for feature in [
    "temperature_available",
    "humidity_available",
    "wind_available",
    "precipitation_available"
]:
    percentage = coverage_check[feature].iloc[0] / total * 100
    print(f"{feature}: {percentage:.2f}%")

print(
    "Fire occurrence rate:",
    round(coverage_check["fire_rows"].iloc[0] / total * 100, 4),
    "%"
)

,total_rows,total_cells,earliest_date,latest_date,temperature_available,humidity_available,wind_available,precipitation_available,fire_rows,no_fire_rows,invalid_labels
0,2359751,76121,2024-01-01,2024-01-31,2348839,2348839,2348839,2348839,81,2359670,0


temperature_available: 99.54%
humidity_available: 99.54%
wind_available: 99.54%
precipitation_available: 99.54%
Fire occurrence rate: 0.0034 %


In [7]:
# Step 5 - Examine dataset size and wildfire class distribution

summary = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT cell_id) AS total_cells,
        MIN(valid_time) AS earliest_date,
        MAX(valid_time) AS latest_date,
        COUNT(*) FILTER (WHERE fire_occurred = 1) AS fire_rows,
        COUNT(*) FILTER (WHERE fire_occurred = 0) AS no_fire_rows,
        COUNT(*) FILTER (WHERE fire_occurred IS NULL) AS missing_labels,
        COUNT(*) FILTER (
            WHERE fire_occurred NOT IN (0, 1)
        ) AS invalid_labels
    FROM read_parquet(?)
""", [str(data_path)]).df()

display(summary)

# Calculate percentage of observations with fire
total = summary["total_rows"].iloc[0]
fires = summary["fire_rows"].iloc[0]

print(f"Fire occurrence rate: {fires / total * 100:.4f}%")

,total_rows,total_cells,earliest_date,latest_date,fire_rows,no_fire_rows,missing_labels,invalid_labels
0,83428616,76121,2022-01-01,2024-12-31,11122,83417494,0,0


Fire occurrence rate: 0.0133%


In [8]:
# Step 6 - Check duplicate grid cell / date combinations

duplicates = con.execute("""
    WITH cell_counts AS (
        SELECT
            cell_id,
            valid_time,
            COUNT(*) AS n
        FROM read_parquet(?)
        WHERE valid_time >= '2024-01-01'
          AND valid_time < '2024-02-01'
        GROUP BY cell_id, valid_time
    )
    SELECT
        SUM(n) AS total_rows,
        COUNT(*) AS unique_cell_days,
        SUM(n - 1) AS duplicate_rows,
        MAX(n) AS max_repeats
    FROM cell_counts
""", [str(data_path)]).df()

display(duplicates)

,total_rows,unique_cell_days,duplicate_rows,max_repeats
0,2359751.0,2359751,0.0,1


In [9]:
# Step 7 - Investigate duplicate observations

duplicate_samples = con.execute("""
    SELECT
        cell_id,
        valid_time,
        COUNT(*) AS repeats,
        COUNT(DISTINCT number) AS unique_numbers,
        COUNT(DISTINCT t2m) AS unique_temperatures,
        COUNT(DISTINCT ndvi) AS unique_ndvi,
        MIN(fire_occurred) AS min_fire,
        MAX(fire_occurred) AS max_fire
    FROM read_parquet(?)
    WHERE valid_time >= '2024-01-01'
      AND valid_time < '2024-02-01'
    GROUP BY cell_id, valid_time
    HAVING COUNT(*) > 1
    ORDER BY repeats DESC
    LIMIT 10
""", [str(data_path)]).df()

display(duplicate_samples)

,cell_id,valid_time,repeats,unique_numbers,unique_temperatures,unique_ndvi,min_fire,max_fire


In [10]:
# Step 8 - Measure the extent of duplicate observations

duplicate_summary = con.execute("""
    WITH cell_counts AS (
        SELECT
            cell_id,
            valid_time,
            COUNT(*) AS repeats,
            COUNT(DISTINCT t2m) AS unique_temperatures,
            MAX(fire_occurred) AS fire
        FROM read_parquet(?)
        WHERE valid_time >= '2024-01-01'
          AND valid_time < '2024-02-01'
        GROUP BY cell_id, valid_time
    )
    SELECT
        COUNT(*) FILTER (WHERE repeats > 1) AS affected_cell_days,
        COUNT(DISTINCT cell_id) FILTER (WHERE repeats > 1) AS affected_cells,
        SUM(repeats - 1) AS duplicate_rows,
        COUNT(*) FILTER (
            WHERE repeats > 1 AND unique_temperatures > 1
        ) AS conflicting_weather_groups,
        SUM(repeats * fire) AS original_fire_rows,
        SUM(fire) AS unique_fire_cell_days
    FROM cell_counts
""", [str(data_path)]).df()

display(duplicate_summary)

,affected_cell_days,affected_cells,duplicate_rows,conflicting_weather_groups,original_fire_rows,unique_fire_cell_days
0,0,0,0.0,0,81.0,81.0


In [12]:
# Step 9 - Check environmental feature coverage

coverage = con.execute("""
    WITH daily AS (
        SELECT
            cell_id,
            valid_time,
            MAX(fire_occurred) AS fire,
            COUNT(t2m) AS weather_count,
            COUNT(ndvi) AS vegetation_count,
            COUNT(elevation) AS terrain_count
        FROM read_parquet(?)
        WHERE valid_time >= '2024-01-01'
          AND valid_time < '2024-02-01'
        GROUP BY cell_id, valid_time
    )
    SELECT
        COUNT(*) AS total_cell_days,

        COUNT(*) FILTER (
            WHERE weather_count > 0
        ) AS weather_available,

        COUNT(*) FILTER (
            WHERE vegetation_count > 0
        ) AS vegetation_available,

        COUNT(*) FILTER (
            WHERE terrain_count > 0
        ) AS terrain_available,

        COUNT(*) FILTER (
            WHERE fire = 1
        ) AS fire_cell_days,

        COUNT(*) FILTER (
            WHERE fire = 1 AND weather_count > 0
        ) AS fire_days_with_weather

    FROM daily
""", [str(data_path)]).df()

display(coverage)

# Convert feature coverage into percentages
total = coverage["total_cell_days"].iloc[0]

for col in [
    "weather_available",
    "vegetation_available",
    "terrain_available"
]:
    percentage = coverage[col].iloc[0] / total * 100
    print(f"{col}: {percentage:.2f}%")

,total_cell_days,weather_available,vegetation_available,terrain_available,fire_cell_days,fire_days_with_weather
0,2359751,2348839,2348839,2359751,81,81


weather_available: 99.54%
vegetation_available: 99.54%
terrain_available: 100.00%


In [13]:
# Step 10 - Compare fire occurrence with weather availability

weather_fire_check = con.execute("""
    WITH daily AS (
        SELECT
            cell_id,
            valid_time,
            MAX(fire_occurred) AS fire,
            MAX(CASE WHEN t2m IS NOT NULL THEN 1 ELSE 0 END)
                AS has_weather
        FROM read_parquet(?)
        WHERE valid_time >= '2024-01-01'
          AND valid_time < '2024-02-01'
        GROUP BY cell_id, valid_time
    )
    SELECT
        has_weather,
        COUNT(*) AS total_cell_days,
        SUM(fire) AS fire_cell_days,
        100.0 * SUM(fire) / COUNT(*) AS fire_rate_percent
    FROM daily
    GROUP BY has_weather
    ORDER BY has_weather
""", [str(data_path)]).df()

display(weather_fire_check)

,has_weather,total_cell_days,fire_cell_days,fire_rate_percent
0,0,10912,0.0,0.000000
1,1,2348839,81.0,0.003449


In [24]:
# Step 19 - Prepare corrected dataset for Logistic Regression

import numpy as np
import pandas as pd

query = """
WITH daily AS (
    SELECT
        cell_id,
        CAST(valid_time AS DATE) AS weather_date,
        t2m,
        rh2m,
        wind_speed_10m,
        tp,
        ndvi,
        evi,
        elevation,
        slope,
        aspect,
        fire_occurred
    FROM read_parquet(?)
    WHERE t2m IS NOT NULL
),

next_day AS (
    SELECT
        a.cell_id,
        a.weather_date,
        b.weather_date AS valid_time,

        a.t2m,
        a.rh2m,
        a.wind_speed_10m,
        a.tp,
        a.ndvi,
        a.evi,
        a.elevation,
        a.slope,
        a.aspect,

        b.fire_occurred
    FROM daily a
    INNER JOIN daily b
        ON a.cell_id = b.cell_id
        AND b.weather_date = a.weather_date + INTERVAL 1 DAY
    WHERE b.weather_date < '2024-01-01'
),

sampled AS (
    SELECT *
    FROM next_day
    WHERE EXTRACT(YEAR FROM valid_time) = 2023
       OR fire_occurred = 1
       OR (
           hash(
               CAST(cell_id AS VARCHAR) || ':' ||
               CAST(valid_time AS VARCHAR)
           ) % 100
       ) < 2
)

SELECT *
FROM sampled
ORDER BY valid_time, cell_id
"""

df = con.execute(
    query,
    [str(data_path)]
).df()

df["valid_time"] = pd.to_datetime(df["valid_time"])
df["weather_date"] = pd.to_datetime(df["weather_date"])

print("Dataset shape:", df.shape)
print("\nClass distribution:")
print(df["fire_occurred"].value_counts())

print("\nDuplicate rows:",
      df.duplicated(["cell_id", "valid_time"]).sum())

display(df.head())

Dataset shape: (28209909, 13)

Class distribution:
fire_occurred
0.0    28202633
1.0        7276
Name: count, dtype: int64

Duplicate rows: 0


,cell_id,weather_date,valid_time,t2m,rh2m,wind_speed_10m,tp,ndvi,evi,elevation,slope,aspect,fire_occurred
0,409,2022-01-01,2022-01-02,261.865540,78.001411,3.082888,0.000035,NaN,NaN,659,89.464157,236.830109,0.0
1,431,2022-01-01,2022-01-02,261.035492,78.330627,3.854872,0.000039,NaN,NaN,576,89.524826,211.778717,0.0
2,451,2022-01-01,2022-01-02,259.924133,79.400047,5.193607,0.000064,NaN,NaN,566,89.130798,54.260223,0.0
3,478,2022-01-01,2022-01-02,259.790375,76.972786,3.345769,0.000065,0.229196,0.109765,232,88.100578,40.292149,0.0
4,522,2022-01-01,2022-01-02,262.139954,77.055847,2.198470,0.000185,0.212067,0.106262,54,35.264389,315.000000,0.0


In [25]:
# Step 12 - Feature selection and chronological split

features = [
    "t2m",
    "rh2m",
    "wind_speed_10m",
    "tp",
    "ndvi",
    "evi",
    "elevation",
    "slope",
    "aspect"
]

# Split by target date
train = df[df["valid_time"].dt.year == 2022].copy()
val = df[df["valid_time"].dt.year == 2023].copy()

X_train = train[features]
y_train = train["fire_occurred"].astype(int)

X_val = val[features]
y_val = val["fire_occurred"].astype(int)

print("Training observations:", len(train))
print("Validation observations:", len(val))

print("\nTraining fire distribution:")
print(y_train.value_counts())

print("\nValidation fire distribution:")
print(y_val.value_counts())

print("\nTraining missing-value rates:")
print(X_train.isna().mean())

assert y_train.nunique() == 2, "Training set needs both classes"
assert y_val.nunique() == 2, "Validation set needs both classes"
assert not df.duplicated(["cell_id", "valid_time"]).any()

Training observations: 554224
Validation observations: 27655685

Training fire distribution:
fire_occurred
0    550616
1      3608
Name: count, dtype: int64

Validation fire distribution:
fire_occurred
0    27652017
1        3668
Name: count, dtype: int64

Training missing-value rates:
t2m               0.000000
rh2m              0.000000
wind_speed_10m    0.000000
tp                0.000000
ndvi              0.035939
evi               0.035939
elevation         0.000000
slope             0.000000
aspect            0.000000
dtype: float64


In [26]:
# Step 13 - Train Logistic Regression (sampling-adjusted baseline)

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import numpy as np

logreg = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Correct approximately for 2% negative sampling
sample_weights = np.where(
    y_train.to_numpy() == 1,
    1.0,
    50.0
)

logreg.fit(
    X_train,
    y_train,
    classifier__sample_weight=sample_weights
)

print("Logistic Regression training complete!")

Logistic Regression training complete!


In [27]:
# Step 14 - Evaluate Logistic Regression

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score,
    roc_auc_score
)

# Generate predicted probabilities
y_prob = logreg.predict_proba(X_val)[:, 1]

# Set provisional recall-focused classification threshold
selected_threshold = 0.002

y_pred = (y_prob >= selected_threshold).astype(int)

print("Selected threshold:", selected_threshold)
print("Predicted fires:", y_pred.sum())

print("Logistic Regression Results")
print("---------------------------")

print(classification_report(
    y_val, y_pred,
    zero_division=0
))

print("PR-AUC:", average_precision_score(y_val, y_prob))
print("ROC-AUC:", roc_auc_score(y_val, y_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred))

print("\nNo-skill PR-AUC baseline:", y_val.mean())

Selected threshold: 0.002
Predicted fires: 245816
Logistic Regression Results
---------------------------
              precision    recall  f1-score   support

           0       1.00      0.99      1.00  27652017
           1       0.01      0.67      0.02      3668

    accuracy                           0.99  27655685
   macro avg       0.50      0.83      0.51  27655685
weighted avg       1.00      0.99      1.00  27655685

PR-AUC: 0.010470180359250457
ROC-AUC: 0.9645637337297026

Confusion Matrix:
[[27408645   243372]
 [    1224     2444]]

No-skill PR-AUC baseline: 0.00013263095815561971


In [28]:
# Step 16 - Evaluate different classification thresholds

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

thresholds = [
    0.0001, 0.0005, 0.001,
    0.002, 0.005, 0.01,
    0.02, 0.05, 0.1, 0.2, 0.5
]

threshold_results = []

for threshold in thresholds:
    pred = (y_prob >= threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(y_val, pred, zero_division=0),
        "Recall": recall_score(y_val, pred, zero_division=0),
        "F1": f1_score(y_val, pred, zero_division=0),
        "Predicted Fires": int(pred.sum())
    })

threshold_df = pd.DataFrame(threshold_results)

display(threshold_df)

,Threshold,Precision,Recall,F1,Predicted Fires
0,0.0001,0.000365,0.972465,0.000730,9770152
1,0.0005,0.002187,0.882770,0.004362,1480827
2,0.0010,0.004924,0.811341,0.009789,604355
3,0.0020,0.009942,0.666303,0.019592,245816
4,0.0050,0.012788,0.242366,0.024295,69517
5,0.0100,0.015171,0.089149,0.025930,21554
6,0.0200,0.028641,0.035169,0.031571,4504
7,0.0500,0.000000,0.000000,0.000000,453
8,0.1000,0.000000,0.000000,0.000000,68
9,0.2000,0.000000,0.000000,0.000000,1


In [29]:
# Step 17 - Inspect predicted probability distribution

probability_summary = pd.DataFrame({
    "y_true": y_val.to_numpy(),
    "y_prob": y_prob
})

display(
    probability_summary.groupby("y_true")["y_prob"]
    .describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99])
)

,count,mean,std,min,25%,50%,75%,90%,99%,max
y_true,,,,,,,,,,
0,27652017.0,0.000170,0.000715,5.258073e-09,0.000023,0.000061,0.000145,0.000310,0.001815,0.204038
1,3668.0,0.004404,0.005516,1.880131e-05,0.001456,0.002829,0.004912,0.009266,0.029767,0.043533


In [30]:
# Step 15 - Save trained model and validation predictions

import joblib

project_root = data_path.parents[2]

models_dir = project_root / "models"
outputs_dir = project_root / "outputs"

models_dir.mkdir(exist_ok=True)
outputs_dir.mkdir(exist_ok=True)

# Save trained model
model_path = models_dir / "logreg_provisional.joblib"
joblib.dump(logreg, model_path)

# Save validation predictions
predictions = val[
    ["cell_id", "weather_date", "valid_time"]
].copy()

predictions["y_true"] = y_val.to_numpy()
predictions["y_prob"] = y_prob
predictions["y_pred"] = y_pred

# Metadata for evaluation
predictions["model"] = "Logistic Regression"
predictions["threshold"] = selected_threshold

predictions_path = (
    outputs_dir / "logreg_validation_predictions.csv"
)

predictions.to_csv(predictions_path, index=False)

print("Saved model:", model_path)
print("Saved predictions:", predictions_path)

display(predictions.head())

Saved model: /Users/amsal/Documents/GitHub/wildfire-prediction-ca/models/logreg_provisional.joblib
Saved predictions: /Users/amsal/Documents/GitHub/wildfire-prediction-ca/outputs/logreg_validation_predictions.csv


,cell_id,weather_date,valid_time,y_true,y_prob,y_pred,model,threshold
554224,352,2022-12-31,2023-01-01,0,0.004910,1,Logistic Regression,0.002
554225,353,2022-12-31,2023-01-01,0,0.002261,1,Logistic Regression,0.002
554226,354,2022-12-31,2023-01-01,0,0.000867,0,Logistic Regression,0.002
554227,355,2022-12-31,2023-01-01,0,0.000478,0,Logistic Regression,0.002
554228,356,2022-12-31,2023-01-01,0,0.000521,0,Logistic Regression,0.002


In [31]:
# Save predictions in compressed Parquet format

parquet_path = outputs_dir / "logreg_validation_predictions.parquet"

predictions.to_parquet(
    parquet_path,
    compression="zstd",
    index=False
)

print("Saved:", parquet_path)

Saved: /Users/amsal/Documents/GitHub/wildfire-prediction-ca/outputs/logreg_validation_predictions.parquet


In [22]:
# To load above parquet
# import pandas as pd

# predictions = pd.read_parquet(
#     "outputs/logreg_validation_predictions.parquet"
# )